# Travel Query Classifier: TRAVEL vs NOT_TRAVEL

This notebook trains a small local classifier that decides whether a user's **first message** should go to the TravelAgentic LangGraph agent (the `extract` node, which calls a paid LLM) or receive a static "out of scope" reply without any LLM call.

It trains and compares two models on the public **CLINC150** intent dataset:

1. **Baseline:** TF-IDF + Logistic Regression (CPU, seconds).
2. **Transfer learning:** pretrained **DistilBERT** fine-tuned with **LoRA** adapters and a trained classification head (GPU recommended).

**Runtime.** For LoRA choose *Runtime → Change runtime type → T4 GPU* (any CUDA GPU works). Without a GPU the notebook still runs top to bottom but skips LoRA, unless you set `TRAIN_LORA = True` in Section 1 (CPU training works but is much slower).

**Data discipline.** `train` fits the models. `validation` selects the LoRA checkpoint, the rejection threshold and the deployed model. `test` is scored once, in Section 9, after every choice is fixed.

**Honesty.** No trained weights or metrics ship with this notebook; every number it prints comes from your own run. CLINC150 is a *proxy* for TravelAgentic traffic, not a sample of it. Read the label policy (Section 3) and the limitations (Section 12) before letting the gate reject real users.

## 1. Environment setup

The install cell pins the libraries this notebook uses and keeps Colab's preinstalled PyTorch, CUDA packages, NumPy, pandas and PyArrow unchanged (`transformers` 5.x needs torch ≥ 2.5). Two choices avoid conflicts with Colab's GPU image:

- `datasets==4.8.5` is the version Colab already ships. `datasets` 5.1.0 requires `pyarrow>=24`, so installing it upgrades PyArrow and breaks Colab's RAPIDS packages (`cudf-cu13 26.6.0 requires pyarrow<24`).
- `torchao` is uninstalled. Colab ships `torchao` 0.10.0, but PEFT rejects any `torchao` older than 0.16.0 and raises `ImportError` inside `get_peft_model()`. PEFT only uses `torchao` for `torchao`-quantized weights, and this notebook quantizes nothing.

In a fresh runtime no restart is needed. If Colab shows a *Restart session* prompt, or the next cell stops with a restart message, choose *Runtime → Restart session* and then *Runtime → Run all*. The install cell is safe to run again.

In [ ]:
# Colab's torchao 0.10.0 makes PEFT raise ImportError in get_peft_model(); this notebook does not use torchao.
%pip uninstall -y -q torchao
# datasets 4.8.5 is Colab's own version: datasets 5.1.0 needs pyarrow>=24, which breaks cudf-cu13 (pyarrow<24).
%pip -q install "transformers==5.19.0" "peft==0.21.2" "accelerate==1.15.0" "datasets==4.8.5" "scikit-learn==1.9.1" "joblib==1.6.0"

In [ ]:
import json
import platform
import re
import shutil
import time
from importlib.util import find_spec
from pathlib import Path

import datasets
import joblib
import numpy as np
import pandas as pd
import peft
import sklearn
import torch
import transformers
from packaging.version import Version
from transformers import set_seed

assert Version(torch.__version__).release[:2] >= (2, 5), "transformers 5.x needs torch >= 2.5"
# PEFT raises ImportError in get_peft_model() for torchao < 0.16; the install cell removes torchao.
assert find_spec("torchao") is None, "torchao is still importable: rerun the install cell, then Runtime → Restart session"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

SEED = 42
set_seed(SEED)  # seeds Python, NumPy and PyTorch (CPU and CUDA)

DATASET_ID = "clinc/clinc_oos"
DATASET_CONFIG = "plus"  # 100 training examples per intent ("small" has 50)
DATASET_REVISION = "155b9c710419136e17307b80d0a13e68cd46b4ec"
BASE_MODEL = "distilbert/distilbert-base-uncased"
BASE_MODEL_REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"

TARGET_TRAVEL_RECALL = 0.98  # validation target used to choose the rejection threshold
MAX_LENGTH = 64              # CLINC150 training queries are at most 33 tokens

HAS_CUDA = torch.cuda.is_available()
TRAIN_LORA = HAS_CUDA  # set to True to fine-tune on CPU (works, but much slower)

OUTPUT_DIR = Path("/content/travel_query_output" if Path("/content").is_dir() else "travel_query_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VERSIONS = {"python": platform.python_version(), "torch": torch.__version__,
            "transformers": transformers.__version__, "peft": peft.__version__,
            "datasets": datasets.__version__, "scikit-learn": sklearn.__version__,
            "joblib": joblib.__version__, "numpy": np.__version__, "pandas": pd.__version__}
DEVICE = f"cuda ({torch.cuda.get_device_name(0)})" if HAS_CUDA else "cpu"
print(json.dumps(VERSIONS, indent=2))
print("Device:", DEVICE, "| TRAIN_LORA:", TRAIN_LORA, "| outputs:", OUTPUT_DIR.resolve())

## 2. Load and inspect the dataset

[CLINC150](https://huggingface.co/datasets/clinc/clinc_oos) (Larson et al., 2019, CC BY 3.0) contains English assistant queries labeled with 150 intents from 10 domains, plus an `oos` (out-of-scope) class. The `plus` configuration has 100 training queries per intent and 250 `oos` training queries; validation has 20 per intent plus 100 `oos`, and test has 30 per intent plus 1,000 `oos`. The columns are `text` and `intent` (a class label). The dataset revision is pinned for reproducibility.

Only training examples are printed while developing; test examples are not inspected.

In [ ]:
raw = datasets.load_dataset(DATASET_ID, DATASET_CONFIG, revision=DATASET_REVISION)
assert set(raw) == {"train", "validation", "test"}, raw
intent_names = raw["train"].features["intent"].names
assert len(intent_names) == 151 and "oos" in intent_names

frames = {}
for split in ("train", "validation", "test"):
    assert raw[split].features["intent"].names == intent_names
    df = raw[split].to_pandas()  # plain pandas avoids datasets' lazy Column objects
    df["intent"] = df["intent"].map(intent_names.__getitem__)
    frames[split] = df
    print(f"{split:<10} {len(df):>6} rows | {df.intent.nunique()} classes | {int((df.intent == 'oos').sum())} oos")

display(frames["train"].sample(8, random_state=SEED))

## 3. Label policy and preprocessing

The Travel Agent (`backend/services/agent/`) collects origin, destination, fly/drive, one-way/round trip, dates, budget, travelers, rental car, layovers, activities and hotel preferences, and is planned to return flights, hotels, rental cars and historical price trends. The binary labels follow those capabilities:

| Binary label | CLINC150 intents | Reason |
| --- | --- | --- |
| **TRAVEL** (1) | `book_flight`, `book_hotel`, `car_rental`, `travel_suggestion` | Requests the agent is built to handle. |
| **NOT_TRAVEL** (0) | The other 145 intents and `oos` | Includes *travel-adjacent* intents the agent has no tool or data for (`flight_status`, `lost_luggage`, `carry_on`, `international_visa`, `vaccines`, `travel_alert`, `exchange_rate`, `weather`, …). They are reported separately in Section 8. |
| **Excluded** | `pto_request` | Most examples are HR leave requests, but some are vacation requests our agent should accept ("how do i book a vacation", "can you schedule a vacation for next week"). Labeling them NOT_TRAVEL would teach the model to reject vacation requests, so the intent is removed from training and metrics. |

Edit the three sets below if the agent's scope changes, then rerun the notebook.

A few texts occur in more than one split once case, whitespace and punctuation are ignored ("hey what's up" vs "hey whats up"); all of them are short NOT_TRAVEL small talk. Training rows that match validation or test, and validation rows that match test, are dropped, so no text is shared between splits and held-out scores are not inflated. The test split itself is left complete.

In [ ]:
TRAVEL_INTENTS = {"book_flight", "book_hotel", "car_rental", "travel_suggestion"}
# Travel-adjacent but unsupported by the agent: labeled NOT_TRAVEL, reported separately.
TRAVEL_ADJACENT_INTENTS = {
    "flight_status", "lost_luggage", "carry_on", "international_visa", "vaccines",
    "travel_alert", "travel_notification", "exchange_rate", "plug_type", "timezone",
    "translate", "international_fees", "weather", "distance", "directions", "uber",
}
# Ambiguous for this application: removed from training and from the binary metrics.
EXCLUDED_INTENTS = {"pto_request"}
LABEL_NAMES = ["NOT_TRAVEL", "TRAVEL"]

unknown = (TRAVEL_INTENTS | TRAVEL_ADJACENT_INTENTS | EXCLUDED_INTENTS) - set(intent_names)
assert not unknown, f"Not CLINC150 intents: {sorted(unknown)}"
assert not TRAVEL_INTENTS & (TRAVEL_ADJACENT_INTENTS | EXCLUDED_INTENTS)


def prepare(df):
    df = df[~df.intent.isin(EXCLUDED_INTENTS)].copy()
    df["label"] = df.intent.isin(TRAVEL_INTENTS).astype(int)
    return df.reset_index(drop=True)


def normalize(text):
    # Key for finding texts shared between splits: ignores case, whitespace and punctuation.
    return " ".join(re.sub(r"[^\w\s]", "", text.lower()).split())


train_df, val_df, test_df = (prepare(frames[s]) for s in ("train", "validation", "test"))
excluded_val_df = frames["validation"][frames["validation"].intent.isin(EXCLUDED_INTENTS)]

test_keys = set(test_df.text.map(normalize))
val_in_test = val_df.text.map(normalize).isin(test_keys)
val_df = val_df[~val_in_test].reset_index(drop=True)
train_in_held_out = train_df.text.map(normalize).isin(test_keys | set(val_df.text.map(normalize)))
train_df = train_df[~train_in_held_out].reset_index(drop=True)
print(f"Dropped {int(train_in_held_out.sum())} training rows that match validation/test "
      f"and {int(val_in_test.sum())} validation rows that match test")
split_keys = [set(df.text.map(normalize)) for df in (train_df, val_df, test_df)]
assert not (split_keys[0] & split_keys[1] or split_keys[0] & split_keys[2] or split_keys[1] & split_keys[2])

for name, df in (("train", train_df), ("validation", val_df)):
    assert set(df.label) == {0, 1}, f"{name} needs both classes"
display(pd.DataFrame({name: df.label.value_counts().rename(dict(enumerate(LABEL_NAMES)))
                      for name, df in (("train", train_df), ("validation", val_df), ("test", test_df))}).T)
display(train_df[train_df.label == 1].sample(8, random_state=SEED)[["text", "intent"]])

## 4. Evaluation helpers

Each model outputs a score `p_travel` = P(TRAVEL). The gate **forwards** a query to the agent when `p_travel >= threshold` and **rejects** it otherwise, so a false rejection is a real travel query that never reaches the agent: the costliest error.

The threshold is chosen on **validation only**: the highest value whose travel recall still meets `TARGET_TRAVEL_RECALL`. It is placed halfway to the next lower score, so tiny numeric differences (GPU vs CPU) do not flip decisions. It is also capped at 0.5, so a query the model rates as more likely TRAVEL is never rejected. Scores between the threshold and 0.5 are *uncertain* and are forwarded.

Metrics treat TRAVEL as the positive class and "forward" as predicting TRAVEL:

- **accuracy, travel precision, travel recall, travel F1** at the chosen threshold;
- **travel false rejection rate** = travel queries rejected / all travel queries (= 1 − recall);
- **non-travel rejection rate** = non-travel queries rejected / all non-travel queries (LLM calls avoided);
- **share of queries rejected**, the **confusion matrix** `[[TN, FP], [FN, TP]]` (rows = true NOT_TRAVEL/TRAVEL, columns = rejected/forwarded), and a 95% Wilson interval for travel recall (validation and test have only 80 and 120 travel queries).

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score


def choose_threshold(y_true, p_travel, min_recall=TARGET_TRAVEL_RECALL):
    # Highest threshold meeting the recall target, moved halfway to the next lower score, capped at 0.5.
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(p_travel, dtype=float)
    if y.shape != p.shape or not (y == 1).any():
        raise ValueError("Need aligned labels/scores and at least one TRAVEL example")
    candidates = np.unique(np.r_[0.0, p])
    recall = np.array([(p[y == 1] >= t).mean() for t in candidates])
    best = candidates[recall >= min_recall].max()  # 0.0 always qualifies
    lower = candidates[candidates < best]
    threshold = (best + lower.max()) / 2 if lower.size else 0.0
    return float(min(threshold, 0.5))


def wilson_interval(successes, total, z=1.96):
    if total == 0:
        return None
    p = successes / total
    centre = (p + z * z / (2 * total)) / (1 + z * z / total)
    half = z * np.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / (1 + z * z / total)
    return [round(float(centre - half), 4), round(float(centre + half), 4)]


def gate_metrics(y_true, p_travel, threshold):
    y = np.asarray(y_true, dtype=int)
    forwarded = (np.asarray(p_travel, dtype=float) >= threshold).astype(int)
    tn, fp, fn, tp = (int(v) for v in confusion_matrix(y, forwarded, labels=[0, 1]).ravel())
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y, forwarded)),
        "travel_precision": float(precision_score(y, forwarded, zero_division=0)),
        "travel_recall": float(recall_score(y, forwarded, zero_division=0)),
        "travel_f1": float(f1_score(y, forwarded, zero_division=0)),
        "travel_false_rejection_rate": fn / (tp + fn),
        "non_travel_rejection_rate": tn / (tn + fp),
        "share_of_queries_rejected": (tn + fn) / len(y),
        "travel_recall_95ci": wilson_interval(tp, tp + fn),
        "false_rejects": fn,
        "confusion_matrix": [[tn, fp], [fn, tp]],
    }


# Check the helpers on hand-computed cases before training anything.
assert np.isclose(choose_threshold([1, 1, 0], [0.9, 0.7, 0.2], 1.0), 0.45)  # halfway between 0.7 and 0.2
assert np.isclose(choose_threshold([1, 1, 0], [0.4, 0.2, 0.1], 0.5), 0.3)   # one miss allowed
assert choose_threshold([1, 1, 0], [0.99, 0.95, 0.8], 1.0) == 0.5           # capped at 0.5
_m = gate_metrics([1, 1, 0, 0], [0.9, 0.3, 0.6, 0.1], 0.5)
assert _m["confusion_matrix"] == [[1, 1], [1, 1]] and _m["travel_false_rejection_rate"] == 0.5
print("Evaluation helpers OK")

## 5. Baseline: TF-IDF + Logistic Regression (CPU)

Word unigrams and bigrams weighted by TF-IDF feed a class-balanced logistic regression. The fine-tuned transformer has to beat this cheap model to justify its inference cost. Only training rows are passed to `.fit()`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

baseline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED),
)
start = time.perf_counter()
baseline.fit(train_df.text.tolist(), train_df.label.to_numpy())
train_seconds = {"baseline": round(time.perf_counter() - start, 1)}

val_y = val_df.label.to_numpy()
val_scores = {"baseline": baseline.predict_proba(val_df.text.tolist())[:, 1]}
thresholds = {"baseline": choose_threshold(val_y, val_scores["baseline"])}
print(f"Baseline trained in {train_seconds['baseline']}s on {len(train_df)} rows")
print(json.dumps(gate_metrics(val_y, val_scores["baseline"], thresholds["baseline"]), indent=2))

## 6. Pretrained model and LoRA configuration

`distilbert-base-uncased` is a 66M-parameter English encoder pretrained on BookCorpus and Wikipedia. `AutoModelForSequenceClassification` adds two **new, randomly initialized** layers on top: `pre_classifier` and `classifier`.

LoRA freezes every pretrained weight and adds trainable rank-8 matrices to the attention query and value projections (`q_lin`, `v_lin`). The new head layers are listed in `modules_to_save`, so they are fully trained and saved with the adapter. Without that, the head would stay random or be lost on save. The assertions below verify that only LoRA and head parameters are trainable.

**Expected load report.** Loading the checkpoint prints a `DistilBertForSequenceClassification LOAD REPORT`. It is expected and is not an error:

- `UNEXPECTED`: `vocab_transform`, `vocab_layer_norm` and `vocab_projector` (5 tensors) are the masked-language-model head used in pretraining. Sequence classification does not use them, so they are dropped.
- `MISSING`: the `pre_classifier` and `classifier` weights and biases (4 tensors) are the new head described above. They are randomly initialized here and then trained through `modules_to_save`.

Any other key in the report means a different checkpoint or architecture was loaded. LoRA is initialized correctly when this cell prints `trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925` and no assertion fails.

This cell does not need a GPU.

In [ ]:
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ID2LABEL = dict(enumerate(LABEL_NAMES))
LABEL2ID = {name: i for i, name in ID2LABEL.items()}


def load_base_model():
    return AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, revision=BASE_MODEL_REVISION, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],                 # DistilBERT attention projections
    modules_to_save=["pre_classifier", "classifier"],  # new head layers: train fully and save
)
set_seed(SEED)  # reproducible head initialization
lora_model = get_peft_model(load_base_model(), lora_config)
lora_model.print_trainable_parameters()

trainable = [name for name, param in lora_model.named_parameters() if param.requires_grad]
assert all("lora_" in n or "modules_to_save" in n for n in trainable), "pretrained weights must stay frozen"
for head in ("pre_classifier", "classifier"):
    assert any(f".{head}.modules_to_save." in n for n in trainable), f"{head} is not trainable"


def to_dataset(df):
    ds = datasets.Dataset.from_dict({"text": df.text.tolist(), "labels": df.label.tolist()})
    return ds.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
                  batched=True, remove_columns=["text"])


@torch.inference_mode()
def lora_scores(model, tok, texts, batch_size=128):
    # P(TRAVEL) for each text, computed in float32 on the model's device.
    model.eval()
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = tok(list(texts[i:i + batch_size]), truncation=True, max_length=MAX_LENGTH,
                    padding=True, return_tensors="pt").to(model.device)
        scores.append(torch.softmax(model(**batch).logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(scores)


train_tok, val_tok = to_dataset(train_df), to_dataset(val_df)
print(train_tok)

## 7. Fine-tuning (supervised training loop)

`Trainer` runs a standard supervised loop: mini-batches of tokenized queries, cross-entropy loss against the binary labels, AdamW updates of the LoRA and head parameters, and a linearly decaying learning rate. It trains for 3 epochs over the full training split, evaluates validation F1 after each epoch, and restores the best epoch at the end. Mixed precision is fp16 on a T4 and bf16 on Ampere or newer GPUs, for training only. During training, `accelerate` wraps the model's `forward` in autocast and leaves that wrapper in place afterwards. The cell removes it, so validation and test scores, the rejection threshold and the exported model all use float32, like the backend.

Without a GPU this cell is skipped unless `TRAIN_LORA = True`. The README records how long training took on CPU.

In [ ]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments


def trainer_metrics(eval_pred):
    logits = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
    predicted = np.argmax(logits, axis=-1)
    return {"f1": f1_score(eval_pred.label_ids, predicted, zero_division=0),
            "travel_recall": recall_score(eval_pred.label_ids, predicted, zero_division=0)}


if TRAIN_LORA:
    use_bf16 = HAS_CUDA and torch.cuda.get_device_capability()[0] >= 8  # T4 (7.5) lacks native bf16
    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / "checkpoints"),
        num_train_epochs=3,
        learning_rate=2e-4,
        weight_decay=0.01,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=128,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        save_total_limit=1,
        bf16=use_bf16,
        fp16=HAS_CUDA and not use_bf16,
        logging_steps=50,
        report_to="none",
        label_names=["labels"],
        seed=SEED,
    )
    trainer = Trainer(
        model=lora_model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=trainer_metrics,
    )
    start = time.perf_counter()
    trainer.train()
    train_seconds["lora"] = round(time.perf_counter() - start, 1)
    # Mixed precision leaves trainer.model's forward wrapped in fp16/bf16 autocast. Remove the wrapper so every
    # score below (and the threshold chosen from it) is float32, like the exported model loaded by the backend.
    lora_model = trainer.accelerator.unwrap_model(trainer.model, keep_fp32_wrapper=False)  # best epoch by validation F1

    history = pd.DataFrame(trainer.state.log_history)
    display(history.dropna(subset=["eval_f1"])[["epoch", "eval_loss", "eval_f1", "eval_travel_recall"]])
    val_scores["lora"] = lora_scores(lora_model, tokenizer, val_df.text.tolist())
    thresholds["lora"] = choose_threshold(val_y, val_scores["lora"])
    print(f"LoRA trained in {train_seconds['lora']}s on {DEVICE}")
    if HAS_CUDA:
        print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
    print(json.dumps(gate_metrics(val_y, val_scores["lora"], thresholds["lora"]), indent=2))
else:
    print("LoRA fine-tuning skipped (no CUDA GPU). Select a GPU runtime, or set TRAIN_LORA = True, and rerun.")

## 8. Validation comparison and model selection

Both models are compared on validation at the default 0.5 cut-off and at their recall-constrained thresholds. The deployed model is selected by a rule fixed **before** test data is used: both thresholds meet the validation recall target, so the model that rejects more NOT_TRAVEL validation queries wins, and ties go to the cheaper baseline.

The error analysis also uses validation only: the travel queries each gate would reject, and how often travel-adjacent and excluded intents are forwarded.

In [ ]:
def score_texts(name, texts):
    texts = list(texts)
    if name == "baseline":
        return baseline.predict_proba(texts)[:, 1]
    return lora_scores(lora_model, tokenizer, texts)


val_metrics = {name: gate_metrics(val_y, scores, thresholds[name]) for name, scores in val_scores.items()}
COLUMNS = ["threshold", "accuracy", "travel_precision", "travel_recall", "travel_f1",
           "travel_false_rejection_rate", "non_travel_rejection_rate", "share_of_queries_rejected",
           "travel_recall_95ci", "false_rejects", "confusion_matrix"]
rows = []
for name, scores in val_scores.items():
    rows.append({"model": name, "operating_point": "0.5", **gate_metrics(val_y, scores, 0.5)})
    rows.append({"model": name, "operating_point": "recall target", **val_metrics[name]})
display(pd.DataFrame(rows).set_index(["model", "operating_point"])[COLUMNS].round(4))

assert all(m["travel_recall"] >= TARGET_TRAVEL_RECALL for m in val_metrics.values())
SELECTED_MODEL = max(val_metrics, key=lambda m: (val_metrics[m]["non_travel_rejection_rate"], m == "baseline"))
print("Selected model (validation rule):", SELECTED_MODEL)

for name, scores in val_scores.items():
    rejected = (val_y == 1) & (scores < thresholds[name])
    print(f"\n{name}: validation TRAVEL queries that would be rejected ({int(rejected.sum())})")
    if rejected.any():
        display(val_df[rejected].assign(p_travel=scores[rejected])[["text", "intent", "p_travel"]].round(4))

probe = pd.concat([val_df[val_df.intent.isin(TRAVEL_ADJACENT_INTENTS)], excluded_val_df], ignore_index=True)
forward_rate = pd.DataFrame({
    f"{name}_forwarded": pd.Series(score_texts(name, probe.text) >= thresholds[name]).groupby(probe.intent).mean()
    for name in val_scores})
print("\nShare of validation queries forwarded, by travel-adjacent / excluded intent:")
display(forward_rate.sort_values(f"{SELECTED_MODEL}_forwarded", ascending=False).round(2))

## 9. Final test evaluation (run once)

This is the first time test data is scored. The selected model and both thresholds were fixed in Section 8. The non-selected model's test scores are shown for comparison only. **Do not change the label policy, thresholds or model choice because of these numbers**; if you need to revise them, treat this test set as used and evaluate on new held-out data.

In [ ]:
test_y = test_df.label.to_numpy()
test_metrics = {name: gate_metrics(test_y, score_texts(name, test_df.text), thresholds[name]) for name in val_scores}
display(pd.DataFrame(test_metrics).T[COLUMNS].round(4))
print("Selected model:", SELECTED_MODEL)
print(json.dumps(test_metrics[SELECTED_MODEL], indent=2))
if test_metrics[SELECTED_MODEL]["travel_recall"] < TARGET_TRAVEL_RECALL:
    print("WARNING: test travel recall is below the validation target. Do not enable automatic rejection.")

## 10. Prediction interface and example predictions

`classify_query` is the interface proposed for the agent. It returns:

- `label`: `TRAVEL`/`NOT_TRAVEL` (argmax at 0.5), or `None` when the model was not consulted;
- `travel_probability`: P(TRAVEL) from the model (uncalibrated) and `confidence` (probability of `label`);
- `route`: `forward` (send to the `extract` node) or `reject` (static reply, no LLM call);
- `reason`: `travel`, `uncertain` (forwarded although NOT_TRAVEL is more likely), `not_travel`, `active_travel_context`, `language_not_validated` or `empty_input`.

Conservative rules come first. A message in an active travel conversation is always forwarded, because follow-ups like "What about tomorrow?" or "yes" need context. Text with non-ASCII characters is forwarded, because the model has only seen English (typographic quotes from phone keyboards are normalized first).

The examples below are hand-written to show behavior. They are **not** a benchmark. The first six are the routing examples from the project requirements; the cell reports any that a model misroutes. Treat a misroute there as a reason not to enable rejection with that model, even if its CLINC150 metrics look good.

In [ ]:
TYPOGRAPHIC_QUOTES = str.maketrans("‘’“”", "''\"\"")


def classify_query(text, score_fn, reject_threshold, active_travel_context=False):
    # score_fn maps a list of texts to an array of P(TRAVEL); reject_threshold comes from validation.
    def result(label, p, route, reason):
        confidence = None if p is None else round(max(p, 1 - p), 4)
        return {"label": label, "travel_probability": None if p is None else round(p, 4),
                "confidence": confidence, "route": route, "reason": reason}

    if active_travel_context:
        return result(None, None, "forward", "active_travel_context")
    if not isinstance(text, str) or not text.strip():
        return result(None, None, "reject", "empty_input")
    text = text.strip().translate(TYPOGRAPHIC_QUOTES)
    if not text.isascii():
        return result(None, None, "forward", "language_not_validated")
    p = float(score_fn([text])[0])
    if p >= 0.5:
        return result("TRAVEL", p, "forward", "travel")
    if p >= reject_threshold:
        return result("NOT_TRAVEL", p, "forward", "uncertain")
    return result("NOT_TRAVEL", p, "reject", "not_travel")


# Routing examples from the project requirements.
REQUIRED_ROUTES = {
    "Plan a trip to Japan": "forward",
    "Find hotels in Chicago": "forward",
    "Help me plan a business trip": "forward",
    "What is Python?": "reject",
    "How do I cook pasta?": "reject",
    "What is the weather?": "reject",
}
EXAMPLES = [
    *REQUIRED_ROUTES,
    # Harder or ambiguous cases: inspect these by hand.
    "Cheapest way to get from Chicago to New York next weekend",
    "I want to go somewhere warm in March for under $400",
    "Can you help me plan a vacation?",
    "Book me a flight to Boston",
    "I need a rental car in Denver",
    "Do I need a visa for Japan?",
    "What’s the weather in Paris during my trip?",
    "帮我计划去巴黎的旅行",  # Chinese: "help me plan a trip to Paris"
]
examples = pd.DataFrame({"query": EXAMPLES})
for name in val_scores:
    results = [classify_query(q, lambda texts, n=name: score_texts(n, texts), thresholds[name]) for q in EXAMPLES]
    examples[f"{name}_p_travel"] = [r["travel_probability"] for r in results]
    examples[f"{name}_route"] = [f"{r['route']} ({r['reason']})" for r in results]
    misrouted = [q for q, r in zip(EXAMPLES, results) if q in REQUIRED_ROUTES and r["route"] != REQUIRED_ROUTES[q]]
    print(f"{name}: requirement examples misrouted: {misrouted or 'none'}")
display(examples)

print(classify_query("What about tomorrow?", lambda texts: score_texts(SELECTED_MODEL, texts),
                     thresholds[SELECTED_MODEL], active_travel_context=True))

## 11. Save, reload and verify the models

Both models are exported to `OUTPUT_DIR/travel_query_gate/`, together with `metadata.json`, which records the selected model, thresholds, label policy, data and model revisions, package versions and the **measured** validation and test metrics of this run. The cell then reloads the files on CPU, as the backend would, checks that they reproduce the in-memory validation scores, and measures single-query CPU latency. The reload check reports the maximum and mean absolute score difference and the number of validation queries whose route changes, lists any such queries, and fails if any score differs by 1e-3 or more or if any validation route changes. Both sides are scored in float32, so only small GPU-versus-CPU kernel differences should remain. A larger difference means the models are not scored the same way (for example, half precision) or the adapter or head was not saved and reloaded correctly.

The LoRA adapter is not a standalone model: loading it also downloads the pinned DistilBERT base weights (about 270 MB). The same load report as in Section 6 is printed again. The adapter then replaces the randomly initialized head with the trained one, which the reload check confirms. Only load `.joblib` files you created yourself, because unpickling can execute code.

In [ ]:
EXPORT_DIR = OUTPUT_DIR / "travel_query_gate"
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)
joblib.dump(baseline, EXPORT_DIR / "baseline.joblib")
if "lora" in val_scores:
    lora_model.save_pretrained(EXPORT_DIR / "lora_adapter")
    tokenizer.save_pretrained(EXPORT_DIR / "lora_adapter")

# Reload from disk on CPU, exactly as a backend process would.
reloaded = {"baseline": joblib.load(EXPORT_DIR / "baseline.joblib")}
reloaded_scorers = {"baseline": lambda texts: reloaded["baseline"].predict_proba(list(texts))[:, 1]}
if "lora" in val_scores:
    reloaded_tokenizer = AutoTokenizer.from_pretrained(EXPORT_DIR / "lora_adapter")
    reloaded["lora"] = PeftModel.from_pretrained(load_base_model(), EXPORT_DIR / "lora_adapter").to("cpu").eval()
    reloaded_scorers["lora"] = lambda texts: lora_scores(reloaded["lora"], reloaded_tokenizer, list(texts))

reload_check, latency_ms = {}, {}
latency_sample = val_df.text.sample(100, random_state=SEED).tolist()
for name, scorer in reloaded_scorers.items():
    scores = scorer(val_df.text)
    diff = np.abs(scores - val_scores[name])
    flipped = (scores >= thresholds[name]) != (val_scores[name] >= thresholds[name])
    reload_check[name] = {"max_abs_score_diff": float(diff.max()), "mean_abs_score_diff": float(diff.mean()),
                          "routing_flips_on_validation": int(flipped.sum())}
    if flipped.any():  # validation queries the exported model would route differently
        display(val_df[flipped].assign(in_memory=val_scores[name][flipped], reloaded=scores[flipped])[["text", "intent", "in_memory", "reloaded"]])
    assert diff.max() < 1e-3 and not flipped.any(), f"{name}: reloaded model differs from the evaluated one: {reload_check[name]}"
    timings = []
    for text in latency_sample:
        start = time.perf_counter()
        scorer([text])
        timings.append(time.perf_counter() - start)
    latency_ms[name] = round(1000 * float(np.median(timings)), 2)
print("Reload check:", json.dumps(reload_check, indent=2))
print("Median single-query CPU latency (ms):", latency_ms)

metadata = {
    "selected_model": SELECTED_MODEL,
    "thresholds": thresholds,
    "target_travel_recall": TARGET_TRAVEL_RECALL,
    "label_names": LABEL_NAMES,
    "travel_intents": sorted(TRAVEL_INTENTS),
    "travel_adjacent_intents": sorted(TRAVEL_ADJACENT_INTENTS),
    "excluded_intents": sorted(EXCLUDED_INTENTS),
    "dataset": {"id": DATASET_ID, "config": DATASET_CONFIG, "revision": DATASET_REVISION,
                "rows": {"train": len(train_df), "validation": len(val_df), "test": len(test_df)}},
    "base_model": {"id": BASE_MODEL, "revision": BASE_MODEL_REVISION},
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "device": DEVICE,
    "train_seconds": train_seconds,
    "validation_metrics": val_metrics,
    "test_metrics": test_metrics,
    "reload_check": reload_check,
    "cpu_latency_ms_median": latency_ms,
    "versions": VERSIONS,
}
(EXPORT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(OUTPUT_DIR / "travel_query_gate"), "zip", root_dir=OUTPUT_DIR, base_dir="travel_query_gate")
print("Exported:", sorted(p.relative_to(EXPORT_DIR).as_posix() for p in EXPORT_DIR.rglob("*") if p.is_file()))
print("Archive:", archive)

# The reloaded selected model behind the same interface:
for query in ("Find hotels in Chicago", "How do I cook pasta?"):
    print(query, "->", classify_query(query, reloaded_scorers[SELECTED_MODEL], thresholds[SELECTED_MODEL]))

# On Colab, uncomment to download the archive (files under /content are deleted when the runtime ends):
# from google.colab import files
# files.download(archive)

## 12. Limitations and next steps

- **Proxy labels.** CLINC150 is a general assistant dataset, not TravelAgentic traffic. Its TRAVEL queries are short, lowercase and single-intent; words central to this app ("plan", "vacation", "budget", "cheap", "itinerary", "business trip") are rare or absent in its TRAVEL examples, and "plan" and "vacation" appear mostly in NOT_TRAVEL intents (insurance plans, PTO). Check the example predictions above before trusting the gate on real phrasing.
- **Small positive counts.** Validation has 80 and test 120 TRAVEL queries, so recall is measured coarsely; the 95% interval is reported for that reason. With 80 positives, a 98% target allows at most one validation miss, so a model can meet the target on validation and miss it on test; Section 9 warns when that happens.
- **Label policy is a product decision.** Travel-adjacent questions (visas, flight status, weather) are labeled NOT_TRAVEL because the agent cannot answer them today. Move intents into `TRAVEL_INTENTS` if the agent's scope grows.
- **English only, single turn.** Non-ASCII text is forwarded, and context is handled only by the `active_travel_context` flag.
- **Scores are not calibrated probabilities.** The threshold is what matters, and it is tied to the model it was chosen for.

Before enabling rejection in production, run the gate in **shadow mode** (log its decision and always forward), label a few hundred real first messages, measure false rejects and LLM calls avoided on them, and retrain or re-threshold on that data. See the README for the proposed integration before the `extract` node.